# V9.1 — Partie 1 : Extraction RAW Qwen

**Objectif : une seule responsabilité : lire les PDF et produire un JSON RAW immuable par dossier.**

Cette partie conserve les capacités de lecture de la V8.1, notamment le **titre/permis de travail en HD (1800 px, sécurité 2000 px)**, les recadrages et les retries ciblés sur champs manquants. Elle ne fait **aucune normalisation métier**, aucun rapprochement inter-document, aucune consolidation métier et aucun `PLANNING_TL`.

Le contrat de sortie est `DOM_EXTRACTION_V1`. Les **99 noms de champs sont strictement identiques à V8.1**.

Pour le test initial, `MAX_PDFS = 10`. Une fois validé, mettre `MAX_PDFS = None`.


## V9.1 — optimisation et transparence des appels Qwen

Cette version conserve la lecture **HD initiale du TITRE_TRAVAIL** (nécessaire pour maintenir la qualité obtenue depuis la V7.2), mais n'effectue **aucune deuxième lecture si les champs critiques sont présents et la première extraction est suffisante**.

Le JSON est aussi allégé : `raw_data` contient le résultat final, et `extraction_attempts` contient une seule trace par véritable appel Qwen. Le champ agrégé `extraction_raw_text` est désactivé par défaut pour éviter la duplication visuelle.


## 1. Dépendances


In [ ]:
# Si nécessaire sur un environnement neuf Domino :
# %pip install -q -U 'transformers>=4.57.0' accelerate pymupdf pillow pandas psutil
#
# IMPORTANT : aucune dépendance flash-attn n'est requise ni utilisée.


## 2. Imports


In [ ]:
import gc
import hashlib
import json
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import fitz
import numpy as np
import pandas as pd
import psutil
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

print('✅ Imports OK')
print('Python      :', sys.version.split()[0])
print('Torch       :', torch.__version__)
print('CUDA dispo :', torch.cuda.is_available())
print('GPU        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun')


## 3. Configuration V9


In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---------- Contrat de données ----------
SCHEMA_VERSION = 'DOM_EXTRACTION_V1'
PIPELINE_VERSION = 'DOM_V9_1_PART1_EXTRACTION_RAW_OPTIMIZED'
FIELD_SCHEMA_HASH = 'da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e'

# ---------- Test ----------
MAX_PDFS = 10       # mettre None après validation sur les 10 dossiers
RESUME = True

# ---------- Images ----------
PDF_ZOOM = 2.0
IMAGE_MAX_SIZE = 1400
IMAGE_MAX_SIZE_CLASSIFICATION = 1100
PDF_ZOOM_HAUTE_DEF = 4
IMAGE_MAX_SIZE_HAUTE_DEF = 1800
IMAGE_MAX_SIZE_PERMIS_RETRY = 2000
MIN_PIXELS = 4 * 32 * 32
MAX_PIXELS = 2200 * 32 * 32
BLANK_THRESHOLD = 0.95
CLASSIFICATION_THRESHOLD = 0.80
CLASSIFICATION_RETRY_ON_AUTRE = True

# ---------- Batch ----------
GPU_BATCH_SIZE_CLASSIFICATION = 16
GPU_BATCH_SIZE_EXTRACTION_STANDARD = 4
GPU_BATCH_SIZE_EXTRACTION_HD = 2
MAX_NEW_TOKENS_CLASSIFICATION = 100
MAX_NEW_TOKENS_EXTRACTION = 1700
MAX_NEW_TOKENS_TARGETED = 650

# ---------- Extraction / retries ----------
SEUIL_REMPLISSAGE_OK = 0.55
SEUIL_REMPLISSAGE_MIN = 0.40
MAX_TARGETED_FIELDS = 14
ENABLE_TARGETED_RETRY = True
ENABLE_FINAL_HD_SAFETY = True
ENABLE_VIRTUAL_PERMIT_COVER = True
VIRTUAL_COVER_MIN_INK_RATIO = 0.025

# ---------- Audit / transparence des appels Qwen ----------
# Le JSON final conserve une seule copie de la réponse brute par appel dans
# extraction_attempts. raw_data contient la donnée finale extraite.
STORE_QWEN_RAW_TEXT_IN_ATTEMPTS = True
STORE_PARSED_DATA_IN_ATTEMPTS = False   # évite de dupliquer raw_data
STORE_AGGREGATED_EXTRACTION_RAW_TEXT = False  # ancien champ très redondant
PRINT_CALL_DIAGNOSTICS = True

# ---------- Fichiers ----------
INPUT_DIR = Path('/mnt/data/domiciliations_in')
OUTPUT_ROOT = Path('/mnt/data/domiciliations_v9')
RAW_ROOT = OUTPUT_ROOT / '01_extraction_raw'
JSON_DIR = RAW_ROOT / 'json_dossiers'
LOG_PATH = RAW_ROOT / 'pipeline_extraction_v9.log'
MANIFEST_PATH = RAW_ROOT / 'extraction_manifest.json'
INDEX_CSV_PATH = RAW_ROOT / 'extraction_index.csv'

INPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(INPUT_DIR.glob('*.pdf'))
if MAX_PDFS is not None:
    pdfs = pdfs[:int(MAX_PDFS)]

print('Pipeline       :', PIPELINE_VERSION)
print('Schema         :', SCHEMA_VERSION)
print('Schema hash    :', FIELD_SCHEMA_HASH[:16] + '…')
print('PDFs sélectionnés :', len(pdfs))
print('Entrée         :', INPUT_DIR)
print('Sortie RAW     :', JSON_DIR)
print('Permis HD      :', IMAGE_MAX_SIZE_HAUTE_DEF, '| retry', IMAGE_MAX_SIZE_PERMIS_RETRY)


## 4. Chargement Qwen — sans Flash-Attention


In [ ]:
if DEVICE != 'cuda':
    raise RuntimeError('Ce pipeline nécessite un GPU CUDA.')

torch.backends.cuda.matmul.allow_tf32 = True

print('Chargement du processor...')
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = 'left'

print('Chargement du modèle FP8...')
print('ℹ️ V9 : aucune dépendance flash-attn / flash_attention_2.')
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

# Chargement volontairement aligné sur la V7.2/V8.1 qui fonctionne sur Domino.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=True),
)
model.eval()

print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')
print(f'VRAM allouée : {torch.cuda.memory_allocated()/1e9:.2f} GB')
_device_map = getattr(model, 'hf_device_map', None)
if _device_map:
    print('Device map    :', _device_map)
    _offload = [v for v in _device_map.values() if str(v).lower() in {'cpu','disk'}]
    if _offload:
        print('⚠️ Offload CPU/disk détecté : inférence potentiellement ralentie.')


## 5. Utilitaires PDF / image / JSON


In [ ]:
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def resize_image(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    ratio = max_side / max(w, h)
    return img.resize((int(w*ratio), int(h*ratio)), Image.LANCZOS)


def white_ratio(image):
    arr = np.array(image.convert('L'))
    return float((arr > 245).sum() / arr.size)


def pdf_to_pages(path, zoom=PDF_ZOOM):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise ValueError(f'PDF absent ou vide : {path}')
    pages=[]
    doc=fitz.open(str(path))
    try:
        matrix=fitz.Matrix(zoom, zoom)
        for i in range(doc.page_count):
            page=doc.load_page(i)
            pix=page.get_pixmap(matrix=matrix, alpha=False)
            img=Image.frombytes('RGB',(pix.width,pix.height),pix.samples)
            img=resize_image(img)
            pages.append({
                'index':i, 'page_num':i+1, 'image':img,
                'width':img.width, 'height':img.height,
                'white_ratio':round(white_ratio(img),6),
            })
    finally:
        doc.close()
    return pages


def parse_json_response(text):
    if not text:
        return {}
    clean=str(text).strip()
    clean=re.sub(r'^```(?:json)?','',clean,flags=re.I).strip()
    clean=re.sub(r'```$','',clean).strip()
    match=re.search(r'\{.*\}',clean,flags=re.S)
    if not match:
        return {}
    candidate=match.group(0)
    for attempt in [candidate, re.sub(r',\s*([}\]])',r'\1',candidate)]:
        try:
            obj=json.loads(attempt)
            return obj if isinstance(obj,dict) else {}
        except Exception:
            pass
    return {}


def log(message):
    line=f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}"
    print(line)
    with open(LOG_PATH,'a',encoding='utf-8') as f:
        f.write(line+'\n')


def crop_region(image, haut=0.0, bas=1.0, gauche=0.0, droite=1.0):
    largeur, hauteur=image.size
    x0=int(max(0,min(1,gauche))*largeur); x1=int(max(0,min(1,droite))*largeur)
    y0=int(max(0,min(1,haut))*hauteur); y1=int(max(0,min(1,bas))*hauteur)
    if x1<=x0 or y1<=y0:
        return image
    return image.crop((x0,y0,x1,y1))


def render_page_region(pdf_path, page_index, zoom=PDF_ZOOM_HAUTE_DEF,
                       max_side=IMAGE_MAX_SIZE_HAUTE_DEF, crop=None):
    doc=fitz.open(str(pdf_path))
    try:
        page=doc.load_page(int(page_index))
        pix=page.get_pixmap(matrix=fitz.Matrix(zoom,zoom),alpha=False)
        img=Image.frombytes('RGB',(pix.width,pix.height),pix.samples)
    finally:
        doc.close()
    if crop:
        img=crop_region(img,*crop)
    return resize_image(img,max_side=max_side)


FRONTIERE_ZONE_RECHERCHE=(0.28,0.66)
FRONTIERE_SEUIL_ENCRE=0.004
FRONTIERE_DEFAUT=0.47

def detecter_frontiere_documents(image, zone=FRONTIERE_ZONE_RECHERCHE,
                                  seuil_encre=FRONTIERE_SEUIL_ENCRE,
                                  defaut=FRONTIERE_DEFAUT):
    try:
        arr=np.array(image.convert('L'))
    except Exception:
        return defaut
    h=arr.shape[0]
    if h<10:
        return defaut
    densite=(arr<200).sum(axis=1)/max(arr.shape[1],1)
    y_min=int(max(0,zone[0])*h); y_max=int(min(1,zone[1])*h)
    bandes=[]; debut=None
    for y in range(y_min,y_max):
        vide=densite[y]<seuil_encre
        if vide and debut is None:
            debut=y
        elif not vide and debut is not None:
            bandes.append((debut,y)); debut=None
    if debut is not None:
        bandes.append((debut,y_max))
    if not bandes:
        return defaut
    debut,fin=max(bandes,key=lambda b:b[1]-b[0])
    if (fin-debut)/h<0.015:
        return defaut
    return round((debut+fin)/2/h,4)


def crops_planche_permis(image,marge=0.02):
    f=detecter_frontiere_documents(image)
    bas=min(1.0,f+marge); haut=max(0.0,f-marge)
    return {
        'frontiere':f,
        'titre':(0.00,bas,0.00,1.00),
        'colonne_identite':(0.00,bas,0.44,1.00),
        'colonne_poste':(0.00,bas,0.00,0.56),
        'couverture':(haut,1.00,0.00,1.00),
    }


def ink_ratio(image,threshold=200):
    arr=np.array(image.convert('L'))
    return float((arr<threshold).sum()/arr.size) if arr.size else 0.0


def image_for_classification(image):
    return resize_image(image,max_side=IMAGE_MAX_SIZE_CLASSIFICATION)


def is_missing_raw(value):
    # Sert uniquement à décider si Qwen doit relire un champ.
    # La valeur stockée dans raw_data n'est jamais normalisée par cette fonction.
    if value is None:
        return True
    if isinstance(value,str):
        t=value.strip()
        return (not t) or t.upper() in {'NULL','NONE','N/A','NA','ILLISIBLE','NON LISIBLE'}
    return False


def value_is_suspect(field,value):
    # V9 : contrôle d'extraction volontairement minimal.
    # Aucun parseur montant/date ici. Les formats sont traités en Partie 2.
    if is_missing_raw(value):
        return True
    text=str(value).strip()
    if field in {'TTR_NUMERO_PERMIS','CTR_NUMERO_PERMIS_TRAVAIL',
                 'CTS_NUMERO_PERMIS_TRAVAIL','PTR_NUMERO_SERIE'}:
        return len(re.sub(r'[^A-Za-z0-9]','',text)) < 5
    if field == 'DOM_COMPTE_LOCAL':
        return len(re.sub(r'\s+','',text)) < 8
    return False


def taux_remplissage(data,champs_attendus):
    if not champs_attendus:
        return 1.0
    n=sum(1 for c in champs_attendus if not is_missing_raw((data or {}).get(c)))
    return round(n/len(champs_attendus),4)

print('✅ Utilitaires V9 RAW OK')


## 6. Inférence GPU batch


In [ ]:
def apply_template(messages):
    """Qwen3 : désactive le mode 'thinking' si supporté."""
    try:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def ask_single(prompt, image, max_new_tokens):
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': prompt},
        ],
    }]

    text_in = apply_template(messages)
    inputs = processor(
        text=[text_in],
        images=[image],
        return_tensors='pt',
    ).to(DEVICE)

    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            use_cache=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()

    generated = out[0][inputs['input_ids'].shape[1]:]
    text = processor.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    return {
        'text': text,
        'tokens_in': int(inputs['input_ids'].shape[1]),
        'tokens_out': int(len(generated)),
        'elapsed_s': round(time.time() - t0, 3),
    }


def ask_batch_mixed(prompts, images, max_new_tokens):
    """
    Batch VLM où chaque image peut avoir son propre prompt.
    C'est la différence essentielle par rapport à la V7.2 : engagement,
    contrat et contrat spécifique peuvent être inférés en parallèle.
    """
    if not images:
        return []
    if len(prompts) != len(images):
        raise ValueError('prompts et images doivent avoir la même longueur')
    if len(images) == 1:
        return [ask_single(prompts[0], images[0], max_new_tokens)]

    texts_in = []
    for prompt, image in zip(prompts, images):
        messages = [{
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': prompt},
            ],
        }]
        texts_in.append(apply_template(messages))

    inputs = processor(
        text=texts_in,
        images=images,
        return_tensors='pt',
        padding=True,
    ).to(DEVICE)

    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            use_cache=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()

    if out.shape[0] != len(images):
        raise RuntimeError(
            f'Réponses VLM incohérentes : {out.shape[0]} sortie(s) '
            f'pour {len(images)} image(s)'
        )

    elapsed = time.time() - t0
    input_width = inputs['input_ids'].shape[1]
    attention_mask = inputs.get('attention_mask')
    results = []

    for i in range(len(images)):
        generated = out[i][input_width:]
        text = processor.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        tokens_in = (
            int(attention_mask[i].sum().item())
            if attention_mask is not None
            else int(input_width)
        )
        results.append({
            'text': text,
            'tokens_in': tokens_in,
            'tokens_out': int(len(generated)),
            'elapsed_s': round(elapsed / len(images), 3),
        })

    return results


def ask_batch(prompt, images, max_new_tokens):
    """Compatibilité avec la classification V7 : même prompt pour le batch."""
    return ask_batch_mixed([prompt] * len(images), images, max_new_tokens)


def is_cuda_oom(exc):
    text = str(exc).lower()
    return isinstance(exc, torch.cuda.OutOfMemoryError) or 'out of memory' in text


print('✅ Inférence V9 single / batch multi-prompts OK')


## 7. Prompts — mêmes 99 champs que V8.1


In [ ]:
PROMPT_CLASSIFICATION = """
Analyse le titre, les en-têtes, la mise en page et les blocs visuels
de cette page.

Classe la page dans exactement une seule catégorie :

- ENGAGEMENT_DOMICILIATION
- CONTRAT_TRAVAIL
- CONTRAT_SPECIFIQUE
- TITRE_TRAVAIL
- PERMIS_TRAVAIL_COUVERTURE
- AUTRE

RÈGLE DE PRIORITÉ ABSOLUE :
Certaines pages contiennent DEUX documents superposés : un titre de
travail bilingue dans la moitié haute et une couverture de permis de
travail dans la moitié basse.
Dans ce cas, classer TOUJOURS la page en TITRE_TRAVAIL.
Le grand titre « جواز العمل / Permis de Travail » de la moitié basse ne
doit JAMAIS l'emporter sur un bloc d'identité présent dans la moitié
haute.

Règles de classification :

- ENGAGEMENT_DOMICILIATION :
  le titre contient « ENGAGEMENT DE DOMICILIATION »
  ou « CONTRAT DES SALARIES ETRANGERS ».

- CONTRAT_TRAVAIL :
  le titre contient « CONTRAT DE TRAVAIL A DUREE DETERMINEE ».

- CONTRAT_SPECIFIQUE :
  le titre contient « CONTRAT DE TRAVAIL SPECIFIQUE
  A LA MAIN D'OEUVRE ETRANGERE ».

- TITRE_TRAVAIL :
  la page contient un bloc d'identité du travailleur, reconnaissable à
  AU MOINS DEUX des éléments suivants :
    - une photographie d'identité ;
    - les libellés « Nom » et « Prénom » suivis de valeurs ;
    - les libellés « Date de naissance » / « Lieu de naissance » ;
    - le libellé « Date d'entrée en Algérie » ;
    - des libellés arabes d'identité (اللقب، الإسم، تاريخ الإزدياد).
  Cette catégorie s'applique même si la page comporte aussi des cachets,
  un QR code, du texte de loi ou un second document en dessous.

- PERMIS_TRAVAIL_COUVERTURE :
  UNIQUEMENT si la page ne contient AUCUN bloc d'identité du travailleur
  et se limite au titre « Permis de Travail », au numéro de série et aux
  extraits de loi.

- AUTRE :
  aucun type ne correspond clairement.

Ne te base jamais uniquement sur le numéro de page.

Retourne uniquement ce JSON :
{
  "type_document": "TYPE",
  "confidence": 0.00,
  "titre_detecte": "TITRE BRUT OU null",
  "bloc_identite_present": true
}
"""

COMMON_RAW_RULES = """
Tu analyses une seule image, qui peut être une page entière ou un
recadrage d'une page.

RÈGLES OBLIGATOIRES :
1. Extraire uniquement les champs demandés.
2. Pour chaque champ, rechercher le libellé indiqué.
3. Recopier uniquement la valeur située juste après le libellé :
   - sur la même ligne ;
   - ou immédiatement sur la ligne suivante si la valeur continue.
4. Conserver la valeur exactement comme elle apparaît :
   espaces, ponctuation, séparateurs, format de date et format de montant.
5. Ne corrige pas l'orthographe.
6. Ne normalise pas les dates.
7. Ne normalise pas les montants.
8. Ne sépare pas automatiquement le nom et le prénom.
9. Ne complète pas une valeur partiellement lisible.
10. N'utilise aucune valeur provenant d'une autre page.
11. Si le libellé est absent ou la valeur illisible, retourne null.
12. N'invente jamais une valeur.
13. Retourne uniquement un objet JSON valide, sans commentaire.
14. Un champ absent du recadrage que tu analyses doit valoir null.
    Ne devine pas ce qui se trouve hors de l'image.
"""

PROMPT_ENGAGEMENT = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION

Extrais exactement les clés suivantes :

{
  "DOM_NOM_RAISON_SOCIAL_CLIENT": null,
  "DOM_COMPTE_LOCAL": null,
  "DOM_ADRESSE_CLIENT": null,
  "DOM_AGENCE_DOMICILIATAIRE": null,
  "DOM_NUMERO_CONTRAT": null,
  "DOM_DUREE_CONTRAT_MOIS": null,
  "DOM_DATE_DEBUT_CONTRAT": null,
  "DOM_DATE_FIN_CONTRAT": null,
  "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": null,
  "DOM_ADRESSE_EMPLOYEUR": null,
  "DOM_SALAIRE_NET_MENSUEL": null,
  "DOM_PART_TRANSFERABLE": null,
  "DOM_TAUX_TRANSFERABLE": null,
  "DOM_MONTANT_TOTAL_DOMICILIE": null,
  "DOM_DATE_SIGNATURE": null
}

Libellés et règles :

- DOM_NOM_RAISON_SOCIAL_CLIENT :
  valeur après « Nom et raison sociale ».
  La valeur peut être répartie en deux colonnes (nom puis prénom) :
  recopier les deux, séparés par un espace.

- DOM_COMPTE_LOCAL :
  valeur après « N de compte » ou « N° de compte ».

- DOM_ADRESSE_CLIENT :
  valeur après la première occurrence de « Adresse »
  dans la section « Identification du client ».

- DOM_AGENCE_DOMICILIATAIRE :
  valeur après « Agence domiciliataire », en haut à droite.

- DOM_NUMERO_CONTRAT :
  valeur après « Numéro du contrat ».

- DOM_DUREE_CONTRAT_MOIS :
  valeur après « Durée du contrat » ou « Duré du contrat ».

- DOM_DATE_DEBUT_CONTRAT :
  valeur après « Date de début de contrat ».

- DOM_DATE_FIN_CONTRAT :
  valeur après « Date de fin de contrat ».

- DOM_NOM_RAISON_SOCIAL_EMPLOYEUR :
  valeur après « Nom et raison sociale de L'Employeur ».

- DOM_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de L'Employeur ».
  Continuer sur la ligne suivante si l'adresse se poursuit.

- DOM_SALAIRE_NET_MENSUEL :
  valeur après « Salaire net mensuel ».

- DOM_PART_TRANSFERABLE :
  valeur après « Montant de la part transférable ».

- DOM_TAUX_TRANSFERABLE :
  valeur après « Pourcentage en regard du salaire net mensuel ».

- DOM_MONTANT_TOTAL_DOMICILIE :
  valeur après « Montant domicilié en DZD ».
  Ce champ est souvent laissé vide : retourner null dans ce cas.

- DOM_DATE_SIGNATURE :
  date manuscrite ou imprimée située près de la mention
  « lu et approuvé », en bas de page.
"""

PROMPT_CONTRAT = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_TRAVAIL

Extrais exactement les clés suivantes :

{
  "CTR_REFERENCE_DOCUMENT": null,
  "CTR_TYPE": null,
  "CTR_EMPLOYEUR": null,
  "CTR_ACTIVITE_EMPLOYEUR": null,
  "CTR_DUREE_MOIS": null,
  "CTR_DATE_DEBUT_CONTRAT": null,
  "CTR_POSTE": null,
  "CTR_NOM_PRENOM_TRAVAILLEUR": null,
  "CTR_PERE_NOM_PRENOM": null,
  "CTR_MERE_NOM_PRENOM": null,
  "CTR_NATIONALITE": null,
  "CTR_DATE_NAISSANCE": null,
  "CTR_LIEU_PAYS_NAISSANCE": null,
  "CTR_ADRESSE_ALGERIE": null,
  "CTR_QUALIFICATION": null,
  "CTR_NUMERO_PERMIS_TRAVAIL": null,
  "CTR_DATE_DELIVRANCE_PERMIS": null,
  "CTR_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTR_DATE_FIN_VALIDITE_PERMIS": null,
  "CTR_SALAIRE_BRUT": null,
  "CTR_SALAIRE_NET": null,
  "CTR_AFFILIATION_SS": null,
  "CTR_NUMERO_EMPLOYEUR": null,
  "CTR_DATE_SIGNATURE": null,
  "CTR_REFERENCE_DOMICILIATION": null,
  "CTR_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTR_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTR_CACHET_EMPLOYEUR_PRESENT": null
}

Libellés et règles :

- CTR_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche,
  par exemple « TR-6166 ».

- CTR_TYPE :
  titre complet du document.

- CTR_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTR_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTR_DUREE_MOIS :
  valeur après « pour une durée de : »
  et avant « à compter du ».

- CTR_DATE_DEBUT_CONTRAT :
  valeur après « à compter du : ».

- CTR_POSTE :
  valeur après « En qualité de : ».

- CTR_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTR_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTR_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTR_NATIONALITE :
  valeur après « Nationalité : ».

- CTR_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTR_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTR_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTR_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTR_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTR_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTR_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « Valable du ».

- CTR_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

- CTR_SALAIRE_BRUT :
  valeur après « Montant du salaire mensuel brut : ».

- CTR_SALAIRE_NET :
  valeur après « Montant du salaire mensuel net : ».

- CTR_AFFILIATION_SS :
  valeur après « Affiliation à la sécurité sociale : ».

- CTR_NUMERO_EMPLOYEUR :
  valeur après « Employeur : ».

- CTR_DATE_SIGNATURE :
  date après « Fait à : Bethioua, le ».

- CTR_REFERENCE_DOMICILIATION :
  dans le cachet « DOMICILIATION IMPORT »,
  recopier les cinq cases dans l'ordre
  et les séparer par « | ».
  Exemple : 271901|2026.1|40|00119|DZD
  null si ce cachet est absent de la page.

Contrôles visuels :
- CTR_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature du Travailleur Etranger », sinon false.

- CTR_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature de l'Employeur », sinon false.

- CTR_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone
  « Signature de l'Employeur », sinon false.
"""

PROMPT_CONTRAT_SPECIFIQUE = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_SPECIFIQUE

Extrais exactement les clés suivantes :

{
  "CTS_REFERENCE_DOCUMENT": null,
  "CTS_SAP_ID": null,
  "CTS_EMPLOYEUR": null,
  "CTS_ACTIVITE_EMPLOYEUR": null,
  "CTS_DUREE_MOIS": null,
  "CTS_DATE_DEBUT_CONTRAT": null,
  "CTS_POSTE": null,
  "CTS_NOM_PRENOM_TRAVAILLEUR": null,
  "CTS_PERE_NOM_PRENOM": null,
  "CTS_MERE_NOM_PRENOM": null,
  "CTS_NATIONALITE": null,
  "CTS_DATE_NAISSANCE": null,
  "CTS_LIEU_PAYS_NAISSANCE": null,
  "CTS_ADRESSE_ALGERIE": null,
  "CTS_QUALIFICATION": null,
  "CTS_NUMERO_PERMIS_TRAVAIL": null,
  "CTS_DATE_DELIVRANCE_PERMIS": null,
  "CTS_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTS_DATE_FIN_VALIDITE_PERMIS": null,
  "CTS_LIGNE_SALAIRE_BRUTE": null,
  "CTS_SALAIRE_NET": null,
  "CTS_SALAIRE_NET_ANCIEN": null,
  "CTS_MENTION_AU_LIEU_DE_PRESENTE": null,
  "CTS_PART_TRANSFERABLE": null,
  "CTS_PART_PAYABLE_DZD": null,
  "CTS_NUMERO_SS_PAYS_ORIGINE": null,
  "CTS_NUMERO_SS_ALGERIE": null,
  "CTS_DATE_DOCUMENT": null,
  "CTS_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTS_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTS_CACHET_EMPLOYEUR_PRESENT": null,
  "CTS_VISA_INSPECTION_TRAVAIL_PRESENT": null
}

Libellés et règles :

- CTS_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche, par exemple « TR-6166 ».

- CTS_SAP_ID :
  valeur après « SAP id - » en haut de page.

- CTS_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTS_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTS_DUREE_MOIS :
  valeur après « pour une durée de : ».

- CTS_DATE_DEBUT_CONTRAT :
  valeur après « A compter du : ».

- CTS_POSTE :
  valeur après « en qualité de : ».

- CTS_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTS_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTS_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTS_NATIONALITE :
  valeur après « Nationalité : ».

- CTS_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTS_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTS_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTS_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTS_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTS_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTS_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « valable du ».

- CTS_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

--- LIGNE DE SALAIRE : RÈGLE PARTICULIÈRE ---

Cette ligne peut prendre DEUX formes :

  Forme A (salaire inchangé) :
    « Salaire mensuel de base net : 506,471.38 »

  Forme B (augmentation de salaire) :
    « Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29 »

- CTS_LIGNE_SALAIRE_BRUTE :
  recopier la ligne ENTIÈRE telle qu'elle apparaît, du libellé
  « Salaire mensuel de base net » jusqu'à la fin de la ligne,
  sans rien retirer.

- CTS_SALAIRE_NET :
  le PREMIER montant de cette ligne, c'est-à-dire celui situé
  immédiatement après « Salaire mensuel de base net : ».
  En forme B, c'est le NOUVEAU salaire, celui placé AVANT
  « au lieu de ». Ne jamais recopier « au lieu de » ni ce qui suit.

- CTS_SALAIRE_NET_ANCIEN :
  le SECOND montant de cette ligne, celui placé APRÈS
  « au lieu de ». C'est l'ANCIEN salaire.
  null si la mention « au lieu de » est absente de cette ligne.

- CTS_MENTION_AU_LIEU_DE_PRESENTE :
  true si la ligne de salaire contient « au lieu de », sinon false.

--- SUITE ---

- CTS_PART_TRANSFERABLE :
  valeur après « La part transférable : ».

- CTS_PART_PAYABLE_DZD :
  valeur après « La part payable en dinars algérien : ».

- CTS_NUMERO_SS_PAYS_ORIGINE :
  valeur après « Dans le pays d'origine : ».

- CTS_NUMERO_SS_ALGERIE :
  valeur après « En Algérie : ».

- CTS_DATE_DOCUMENT :
  date après « Fait à : Bethioua, le ».

Contrôles visuels :
- CTS_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature du Travailleur Etranger ».

- CTS_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature de l'Employeur ».

- CTS_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone employeur.

- CTS_VISA_INSPECTION_TRAVAIL_PRESENT :
  true si le bas de page comporte un cachet ou une mention manuscrite
  près de « Le présent contrat a été visé par nous ».
"""

PROMPT_TITRE_TRAVAIL = COMMON_RAW_RULES + """
TYPE ATTENDU : TITRE_TRAVAIL

DESCRIPTION DE LA PAGE :
Il s'agit d'un titre de travail algérien, bilingue arabe/français,
photocopié en noir et blanc. La qualité est dégradée, les valeurs sont
souvent inscrites sur des lignes de pointillés, et des cachets ronds
peuvent recouvrir partiellement le texte.

MISE EN PAGE — À LIRE ATTENTIVEMENT :
Le document est organisé en DEUX COLONNES.

  COLONNE DE GAUCHE — le poste et l'employeur.
  Chaque ligne suit le schéma :
      libellé français ..... valeur .....        libellé arabe
  Le libellé arabe est collé au bord DROIT de la colonne gauche.
  La valeur se trouve ENTRE le libellé français et le libellé arabe.
  Libellés : « Durée », « Du », « Au », « Lieu de travail »,
  « Nom de l'organisme employeur », « Adresse de l'organisme employeur »,
  « Fait à », « Le ».

  COLONNE DE DROITE — l'identité du travailleur, à côté de la photo.
  Chaque ligne suit le schéma :
      libellé français ..... valeur .....        libellé arabe
  Libellés : « Nom », « Prénom », « Date de naissance »,
  « Lieu de naissance », « Pays », « Nationalité », « Qualification »,
  « Date d'entrée en Algérie ».

RÈGLE CRITIQUE :
Ne JAMAIS confondre un libellé arabe avec une valeur.
La valeur est toujours le texte latin situé sur les pointillés,
entre le libellé français et le libellé arabe.
Si une ligne ne contient que des libellés et des pointillés vides,
retourner null pour ce champ.

Extrais exactement :

{
  "TTR_NUMERO_PERMIS": null,
  "TTR_NUMERO_MANUSCRIT": null,
  "TTR_POSTE": null,
  "TTR_DUREE": null,
  "TTR_DATE_DEBUT": null,
  "TTR_DATE_FIN": null,
  "TTR_LIEU_TRAVAIL": null,
  "TTR_EMPLOYEUR": null,
  "TTR_ADRESSE_EMPLOYEUR": null,
  "TTR_FAIT_A": null,
  "TTR_DATE_DELIVRANCE": null,
  "TTR_NOM": null,
  "TTR_PRENOM": null,
  "TTR_DATE_NAISSANCE": null,
  "TTR_LIEU_NAISSANCE": null,
  "TTR_PAYS": null,
  "TTR_NATIONALITE": null,
  "TTR_QUALIFICATION": null,
  "TTR_DATE_ENTREE_ALGERIE": null,
  "TTR_PHOTO_PRESENTE": null,
  "TTR_CACHET_PRESENT": null
}

Libellés et règles :

- TTR_NUMERO_PERMIS :
  référence encadrée en haut à gauche, de la forme
  « ( R ) 21-00002974 / 31-25-001448 ».
  Recopier la référence complète, y compris les deux parties
  séparées par « / ». Ignorer les parenthèses et la lettre isolée.

- TTR_NUMERO_MANUSCRIT :
  nombre manuscrit inscrit juste sous la référence encadrée,
  par exemple « 6466 ». null si absent.

- TTR_POSTE :
  texte situé sous la phrase
  « Le titulaire du présent permis de travail est autorisé à occuper
  le poste de travail de ».
  Ce texte peut s'étendre sur deux ou trois lignes de pointillés :
  recopier l'ensemble, en séparant les fragments par un espace.

- TTR_DUREE :
  valeur après « Durée », par exemple « 2 ANS, 0 JOURS ».

- TTR_DATE_DEBUT :
  valeur après « Du », sur la ligne portant le libellé arabe
  « إبتداء من ».

- TTR_DATE_FIN :
  valeur après « Au », sur la ligne portant le libellé arabe « إلى ».
  Attention : ce libellé est « Au », pas « Fin de travail ».

- TTR_LIEU_TRAVAIL :
  valeur après « Lieu de travail ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_EMPLOYEUR :
  valeur après « Nom de l'organisme employeur ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de l'organisme employeur ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_FAIT_A :
  valeur après « Fait à ».

- TTR_DATE_DELIVRANCE :
  valeur après « Le », sous « Fait à ».
  Cette date est fréquemment recouverte par un cachet rond :
  si elle reste illisible, retourner null plutôt que de deviner.

- TTR_NOM :
  valeur après « Nom », ligne portant le libellé arabe « اللقب ».
  C'est le nom de famille seul.

- TTR_PRENOM :
  valeur après « Prénom », ligne portant le libellé arabe « الإسم ».

- TTR_DATE_NAISSANCE :
  valeur après « Date de naissance »,
  ligne portant le libellé arabe « تاريخ الإزدياد ».

- TTR_LIEU_NAISSANCE :
  valeur après « Lieu de naissance »,
  ligne portant le libellé arabe « مكان الإزدياد ».
  La valeur peut associer une ville et un pays séparés par « / » :
  recopier l'ensemble.

- TTR_PAYS :
  valeur après « Pays », ligne portant le libellé arabe « البلد ».

- TTR_NATIONALITE :
  valeur après « Nationalité »,
  ligne portant le libellé arabe « الجنسية ».

- TTR_QUALIFICATION :
  valeur après « Qualification »,
  ligne portant le libellé arabe « التأهيل ».
  La valeur peut être coupée en fin de ligne et se poursuivre sur la
  ligne suivante : recopier l'ensemble sans ajouter d'espace au point
  de coupure si le mot est manifestement scindé.

- TTR_DATE_ENTREE_ALGERIE :
  valeur après « Date d'entrée en Algérie »,
  ligne portant le libellé arabe « تاريخ الدخول إلى الجزائر ».

Contrôles visuels :
- TTR_PHOTO_PRESENTE :
  true si une photographie d'identité est visible en haut à droite.

- TTR_CACHET_PRESENT :
  true si au moins un cachet rond est visible sur le document.
"""

PROMPT_PERMIS_COUVERTURE = COMMON_RAW_RULES + """
TYPE ATTENDU : PERMIS_TRAVAIL_COUVERTURE

Cette image correspond à la couverture du permis de travail :
titre « جواز العمل / Permis de Travail », extraits de loi et cachet
de la Direction de l'Emploi de la Wilaya.

Extrais exactement :

{
  "PTR_NUMERO_SERIE": null,
  "PTR_WILAYA": null,
  "PTR_CACHET_DIRECTION_EMPLOI_PRESENT": null
}

- PTR_NUMERO_SERIE :
  numéro de série imprimé en bas de la couverture,
  après « N° de Série » ou isolé en bas à gauche.
  null si illisible.

- PTR_WILAYA :
  valeur après « Direction de l'Emploi de la Wilaya de : ».
  null si la ligne est vide.

- PTR_CACHET_DIRECTION_EMPLOI_PRESENT :
  true si un cachet officiel est visible sur cette zone.

Ne pas extraire le contenu des extraits de loi imprimés à droite.
"""

PROMPTS_EXTRACTION = {
    "ENGAGEMENT_DOMICILIATION": PROMPT_ENGAGEMENT,
    "CONTRAT_TRAVAIL": PROMPT_CONTRAT,
    "CONTRAT_SPECIFIQUE": PROMPT_CONTRAT_SPECIFIQUE,
    "TITRE_TRAVAIL": PROMPT_TITRE_TRAVAIL,
    "PERMIS_TRAVAIL_COUVERTURE": PROMPT_PERMIS_COUVERTURE,
}

TYPES_VALIDES = set(PROMPTS_EXTRACTION) | {"AUTRE"}


def champs_attendus_depuis_prompt(prompt):
    """
    Récupère la liste des clés depuis le squelette JSON du prompt.
    Évite de maintenir une seconde liste qui divergerait des prompts.
    """
    match = re.search(r"\{[^{}]*\}", prompt, flags=re.S)
    if not match:
        return []
    bloc = match.group(0)
    try:
        return list(json.loads(bloc).keys())
    except Exception:
        return re.findall(r'"([A-Z0-9_]+)"\s*:', bloc)


CHAMPS_ATTENDUS = {
    doc_type: champs_attendus_depuis_prompt(prompt)
    for doc_type, prompt in PROMPTS_EXTRACTION.items()
}

# ---------------------------------------------------------------------
# V8.1 - champs critiques et stratégies de relecture
# ---------------------------------------------------------------------

CRITICAL_FIELDS = {
    'ENGAGEMENT_DOMICILIATION': {
        'DOM_NOM_RAISON_SOCIAL_CLIENT', 'DOM_COMPTE_LOCAL',
        'DOM_NUMERO_CONTRAT', 'DOM_DATE_DEBUT_CONTRAT',
        'DOM_DATE_FIN_CONTRAT', 'DOM_SALAIRE_NET_MENSUEL',
        'DOM_PART_TRANSFERABLE',
    },
    'CONTRAT_TRAVAIL': {
        'CTR_NOM_PRENOM_TRAVAILLEUR', 'CTR_NUMERO_PERMIS_TRAVAIL',
        'CTR_DATE_DEBUT_VALIDITE_PERMIS', 'CTR_DATE_FIN_VALIDITE_PERMIS',
        'CTR_SALAIRE_NET',
    },
    'CONTRAT_SPECIFIQUE': {
        'CTS_NOM_PRENOM_TRAVAILLEUR', 'CTS_NUMERO_PERMIS_TRAVAIL',
        'CTS_DATE_DEBUT_VALIDITE_PERMIS', 'CTS_DATE_FIN_VALIDITE_PERMIS',
        'CTS_SALAIRE_NET', 'CTS_PART_TRANSFERABLE',
    },
    'TITRE_TRAVAIL': {
        'TTR_NUMERO_PERMIS', 'TTR_DATE_DEBUT', 'TTR_DATE_FIN',
        'TTR_EMPLOYEUR', 'TTR_NOM', 'TTR_PRENOM',
        'TTR_DATE_NAISSANCE', 'TTR_NATIONALITE',
        'TTR_DATE_ENTREE_ALGERIE',
    },
    'PERMIS_TRAVAIL_COUVERTURE': {
        'PTR_NUMERO_SERIE',
    },
}

TITLE_IDENTITY_FIELDS = {
    'TTR_NOM', 'TTR_PRENOM', 'TTR_DATE_NAISSANCE',
    'TTR_LIEU_NAISSANCE', 'TTR_PAYS', 'TTR_NATIONALITE',
    'TTR_QUALIFICATION', 'TTR_DATE_ENTREE_ALGERIE',
    'TTR_PHOTO_PRESENTE', 'TTR_CACHET_PRESENT',
}

TITLE_POST_FIELDS = set(CHAMPS_ATTENDUS['TITRE_TRAVAIL']) - TITLE_IDENTITY_FIELDS

# Conservé comme documentation et pour les tests. Le moteur V8 construit
# ses jobs à partir de ces profils mais n'exécute plus toutes les stratégies
# séquentiellement par défaut.
STRATEGIES_EXTRACTION = {
    'TITRE_TRAVAIL': [
        {'nom': 'HD_BLOC_TITRE', 'crop_dynamique': 'titre'},
        {'nom': 'HD_COLONNE_IDENTITE', 'crop_dynamique': 'colonne_identite'},
        {'nom': 'HD_COLONNE_POSTE', 'crop_dynamique': 'colonne_poste'},
        {'nom': 'HD_SAFETY_TITRE', 'crop_dynamique': 'titre'},
    ],
    'PERMIS_TRAVAIL_COUVERTURE': [
        {'nom': 'HD_BLOC_COUVERTURE', 'crop_dynamique': 'couverture'},
        {'nom': 'PAGE_ENTIERE_HD'},
    ],
    'ENGAGEMENT_DOMICILIATION': [
        {'nom': 'STANDARD'}, {'nom': 'TARGETED_HD_FULL'},
    ],
    'CONTRAT_TRAVAIL': [
        {'nom': 'STANDARD'}, {'nom': 'TARGETED_HD_CENTRAL'},
    ],
    'CONTRAT_SPECIFIQUE': [
        {'nom': 'STANDARD'}, {'nom': 'TARGETED_HD_CENTRAL'},
    ],
}

# Indications courtes utilisées uniquement dans les mini-prompts de retry.
FIELD_HINTS = {
    # Engagement
    'DOM_NOM_RAISON_SOCIAL_CLIENT': 'valeur après « Nom et raison sociale » dans Identification du client',
    'DOM_COMPTE_LOCAL': 'valeur après « N de compte » ou « N° de compte »',
    'DOM_NUMERO_CONTRAT': 'valeur après « Numéro du contrat »',
    'DOM_DATE_DEBUT_CONTRAT': 'valeur après « Date de début de contrat »',
    'DOM_DATE_FIN_CONTRAT': 'valeur après « Date de fin de contrat »',
    'DOM_SALAIRE_NET_MENSUEL': 'valeur après « Salaire net mensuel »',
    'DOM_PART_TRANSFERABLE': 'valeur après « Montant de la part transférable »',
    # Contrat
    'CTR_NOM_PRENOM_TRAVAILLEUR': 'valeur après « A (Mr/Mme) »',
    'CTR_NUMERO_PERMIS_TRAVAIL': 'référence complète après « permis de travail N° »',
    'CTR_DATE_DEBUT_VALIDITE_PERMIS': 'première date après « Valable du »',
    'CTR_DATE_FIN_VALIDITE_PERMIS': 'date après « au » sur la ligne de validité',
    'CTR_SALAIRE_NET': 'valeur après « Montant du salaire mensuel net »',
    # Contrat spécifique
    'CTS_NOM_PRENOM_TRAVAILLEUR': 'valeur après « A (Mr/Mme) »',
    'CTS_NUMERO_PERMIS_TRAVAIL': 'référence complète après « permis de travail N° »',
    'CTS_DATE_DEBUT_VALIDITE_PERMIS': 'première date après « valable du »',
    'CTS_DATE_FIN_VALIDITE_PERMIS': 'date après « au » sur la ligne de validité',
    'CTS_SALAIRE_NET': 'PREMIER montant après « Salaire mensuel de base net », avant « au lieu de »',
    'CTS_SALAIRE_NET_ANCIEN': 'SECOND montant situé après « au lieu de »',
    'CTS_PART_TRANSFERABLE': 'valeur après « La part transférable »',
    # Titre de travail - colonne gauche / haut
    'TTR_NUMERO_PERMIS': 'référence encadrée en haut à gauche, complète avec les deux parties séparées par /',
    'TTR_NUMERO_MANUSCRIT': 'nombre manuscrit juste sous la référence encadrée',
    'TTR_POSTE': 'texte du poste sous la phrase « autorisé à occuper le poste de travail de »',
    'TTR_DUREE': 'valeur après « Durée »',
    'TTR_DATE_DEBUT': 'valeur après « Du »',
    'TTR_DATE_FIN': 'valeur après « Au »',
    'TTR_LIEU_TRAVAIL': 'valeur après « Lieu de travail »',
    'TTR_EMPLOYEUR': 'valeur après « Nom de l’organisme employeur »',
    'TTR_ADRESSE_EMPLOYEUR': 'valeur après « Adresse de l’organisme employeur »',
    'TTR_FAIT_A': 'valeur après « Fait à »',
    'TTR_DATE_DELIVRANCE': 'date après « Le » sous « Fait à »',
    # Titre de travail - colonne droite / identité
    'TTR_NOM': 'valeur latine après « Nom », avant le libellé arabe',
    'TTR_PRENOM': 'valeur latine après « Prénom », avant le libellé arabe',
    'TTR_DATE_NAISSANCE': 'valeur après « Date de naissance »',
    'TTR_LIEU_NAISSANCE': 'valeur après « Lieu de naissance »',
    'TTR_PAYS': 'valeur après « Pays »',
    'TTR_NATIONALITE': 'valeur après « Nationalité »',
    'TTR_QUALIFICATION': 'valeur après « Qualification »',
    'TTR_DATE_ENTREE_ALGERIE': 'valeur après « Date d’entrée en Algérie »',
    'TTR_PHOTO_PRESENTE': 'true si la photo d’identité est visible, sinon false',
    'TTR_CACHET_PRESENT': 'true si un cachet rond est visible, sinon false',
    # Couverture
    'PTR_NUMERO_SERIE': 'numéro après « N° de Série » sur la couverture Permis de Travail',
    'PTR_WILAYA': 'valeur après « Direction de l’Emploi de la Wilaya de »',
    'PTR_CACHET_DIRECTION_EMPLOI_PRESENT': 'true si le cachet officiel est visible',
}


def build_targeted_prompt(doc_type, fields):
    """Mini-prompt : uniquement les champs à relire/corriger."""
    fields = [f for f in fields if f in CHAMPS_ATTENDUS.get(doc_type, [])]
    skeleton = {f: None for f in fields}
    hints = []
    for field in fields:
        hint = FIELD_HINTS.get(field)
        if hint:
            hints.append(f'- {field}: {hint}.')
        else:
            readable = field.split('_', 1)[-1].replace('_', ' ').lower()
            hints.append(f'- {field}: rechercher le champ « {readable} » sur ce document.')

    return f"""
Analyse uniquement ce recadrage du document {doc_type}.

OBJECTIF : relire uniquement les champs listés ci-dessous.
- Recopie la valeur exactement comme elle apparaît.
- Ne normalise ni date ni montant.
- N'utilise aucune autre page.
- Si la valeur est absente ou illisible : null.
- N'invente jamais.
- Retourne uniquement un JSON valide avec exactement ces clés.

INDICES :
{chr(10).join(hints)}

JSON ATTENDU :
{json.dumps(skeleton, ensure_ascii=False, indent=2)}
"""


def critical_problems(record):
    doc_type = record.get('doc_type')
    data = record.get('raw_data') or {}
    problems = []
    for field in CRITICAL_FIELDS.get(doc_type, set()):
        value = data.get(field)
        if value in (None, '') or value_is_suspect(field, value):
            problems.append(field)
    return sorted(problems)


print('✅ Prompts V9 chargés — schéma des champs identique V8.1')
print('   Types gérés          :', ', '.join(sorted(PROMPTS_EXTRACTION)))
print('   Champs TITRE_TRAVAIL :', len(CHAMPS_ATTENDUS['TITRE_TRAVAIL']))
print('   Champs critiques TITRE:', len(CRITICAL_FIELDS['TITRE_TRAVAIL']))


## 8. Vérification du contrat de schéma


In [ ]:
FIELD_SCHEMA = {k:list(v) for k,v in CHAMPS_ATTENDUS.items()}
_current_hash = hashlib.sha256(
    json.dumps(FIELD_SCHEMA, sort_keys=True, ensure_ascii=False).encode('utf-8')
).hexdigest()
assert _current_hash == FIELD_SCHEMA_HASH, (_current_hash, FIELD_SCHEMA_HASH)
assert sum(len(v) for v in FIELD_SCHEMA.values()) == 99
print('✅ Schéma exact vérifié : 99 champs | hash', FIELD_SCHEMA_HASH[:16]+'…')
for k,v in FIELD_SCHEMA.items():
    print(f'   {k:28} : {len(v):2d} champs')


## 9. Checkpoints RAW V9


In [ ]:
def canonical_checkpoint_path(pdf_path):
    return JSON_DIR / f'{pdf_path.stem}.json'


def checkpoint_is_complete(dossier,pdf_path):
    if not isinstance(dossier,dict):
        return False
    if dossier.get('schema_version') != SCHEMA_VERSION:
        return False
    if dossier.get('pipeline_version') != PIPELINE_VERSION:
        return False
    if dossier.get('field_schema_hash') != FIELD_SCHEMA_HASH:
        return False
    if dossier.get('source_file') != pdf_path.name:
        return False
    if not dossier.get('page_records'):
        return False
    try:
        return dossier.get('source_sha256') == sha256_file(pdf_path)
    except Exception:
        return False


def load_existing_checkpoint(pdf_path):
    p=canonical_checkpoint_path(pdf_path)
    if not (RESUME and p.exists()):
        return None
    try:
        d=json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        return None
    return d if checkpoint_is_complete(d,pdf_path) else None


## 10. Moteur d’extraction RAW


In [ ]:
def _new_record_from_page(page, parsed, output):
    doc_type=parsed.get('type_document') or parsed.get('type') or 'AUTRE'
    try:
        confidence=float(parsed.get('confidence',0) or 0)
    except Exception:
        confidence=0.0
    bloc_identite=bool(parsed.get('bloc_identite_present'))
    if doc_type not in TYPES_VALIDES:
        doc_type='AUTRE'
    requalifie=False
    if bloc_identite and doc_type in ('PERMIS_TRAVAIL_COUVERTURE','AUTRE'):
        doc_type='TITRE_TRAVAIL'; confidence=max(confidence,CLASSIFICATION_THRESHOLD); requalifie=True
    if confidence<CLASSIFICATION_THRESHOLD and not requalifie:
        doc_type='AUTRE'
    return {
        'page_num':page['page_num'],'width':page['width'],'height':page['height'],
        'white_ratio':page['white_ratio'],'image':page['image'],
        'doc_type':doc_type,'titre_detecte':parsed.get('titre_detecte'),
        'bloc_identite_present':bloc_identite,'classification_requalifiee':requalifie,
        'classification_retry_fullres':False,'classification_confidence':confidence,
        'classification_raw_text':output.get('text'),
        'classification_attempts':[{
            'strategy':'LOWRES_1100','raw_text':output.get('text'),
            'parsed':parsed,'tokens_in':output.get('tokens_in',0),
            'tokens_out':output.get('tokens_out',0),'elapsed_s':output.get('elapsed_s',0),
        }],
        'classification_tokens_in':int(output.get('tokens_in',0) or 0),
        'classification_tokens_out':int(output.get('tokens_out',0) or 0),
        'classification_elapsed_s':float(output.get('elapsed_s',0) or 0),
        'raw_data':{},'extraction_status':'NON_LANCEE','extraction_error':None,
        'extraction_raw_text':None,'extraction_attempts':[],
        'extraction_strategies':[],'extraction_taux_remplissage':0.0,
        'extraction_call_count':0,'retry_call_count':0,
        'extraction_tokens_in':0,'extraction_tokens_out':0,'extraction_elapsed_s':0.0,
        'field_revisions':[],'critical_fields_missing':[],'quality_flags':[],
        'is_virtual_subdocument':False,
    }


def classify_pages(pages):
    records=[]
    for start in range(0,len(pages),GPU_BATCH_SIZE_CLASSIFICATION):
        batch=pages[start:start+GPU_BATCH_SIZE_CLASSIFICATION]
        outs=ask_batch(PROMPT_CLASSIFICATION,[image_for_classification(x['image']) for x in batch],MAX_NEW_TOKENS_CLASSIFICATION)
        for page,out in zip(batch,outs):
            records.append(_new_record_from_page(page,parse_json_response(out['text']),out))

    if CLASSIFICATION_RETRY_ON_AUTRE:
        retry=[r for r in records if r['doc_type']=='AUTRE']
        for start in range(0,len(retry),GPU_BATCH_SIZE_CLASSIFICATION):
            batch=retry[start:start+GPU_BATCH_SIZE_CLASSIFICATION]
            outs=ask_batch(PROMPT_CLASSIFICATION,[r['image'] for r in batch],MAX_NEW_TOKENS_CLASSIFICATION)
            for r,out in zip(batch,outs):
                parsed=parse_json_response(out['text'])
                attempt={'strategy':'FULLRES_1400_RETRY','raw_text':out.get('text'),'parsed':parsed,
                         'tokens_in':out.get('tokens_in',0),'tokens_out':out.get('tokens_out',0),'elapsed_s':out.get('elapsed_s',0)}
                new=_new_record_from_page({'page_num':r['page_num'],'width':r['width'],'height':r['height'],
                                           'white_ratio':r['white_ratio'],'image':r['image']},parsed,out)
                attempts=list(r.get('classification_attempts') or [])+[attempt]
                new['classification_attempts']=attempts
                new['classification_tokens_in']=sum(int(a.get('tokens_in',0) or 0) for a in attempts)
                new['classification_tokens_out']=sum(int(a.get('tokens_out',0) or 0) for a in attempts)
                new['classification_elapsed_s']=round(sum(float(a.get('elapsed_s',0) or 0) for a in attempts),3)
                new['classification_raw_text']='\n\n'.join(f"[{a['strategy']}] {a.get('raw_text','')}" for a in attempts)
                new['classification_retry_fullres']=True
                r.clear(); r.update(new)
    return records


def add_virtual_permit_subdocuments(records):
    if not ENABLE_VIRTUAL_PERMIT_COVER:
        return records
    if any(r.get('doc_type')=='PERMIS_TRAVAIL_COUVERTURE' for r in records):
        return records
    max_page=max((r.get('page_num',0) for r in records),default=0)
    additions=[]
    for r in records:
        if r.get('doc_type')!='TITRE_TRAVAIL':
            continue
        crops=crops_planche_permis(r['image'])
        lower=crop_region(r['image'],*crops['couverture'])
        lower_ink=ink_ratio(lower)
        if not (r.get('page_num')==max_page or lower_ink>=VIRTUAL_COVER_MIN_INK_RATIO):
            continue
        v={k:r[k] for k in ['page_num','width','height','white_ratio','image']}
        v.update({
            'doc_type':'PERMIS_TRAVAIL_COUVERTURE','titre_detecte':'SOUS_DOCUMENT_DERIVE_DE_TITRE_TRAVAIL',
            'bloc_identite_present':False,'classification_requalifiee':False,
            'classification_retry_fullres':False,'classification_confidence':r.get('classification_confidence',0),
            'classification_raw_text':'DERIVE_SANS_NOUVELLE_CLASSIFICATION_VLM','classification_attempts':[],
            'classification_tokens_in':0,'classification_tokens_out':0,'classification_elapsed_s':0.0,
            'raw_data':{},'extraction_status':'NON_LANCEE','extraction_error':None,
            'extraction_raw_text':None,'extraction_attempts':[],'extraction_strategies':[],
            'extraction_taux_remplissage':0.0,'extraction_tokens_in':0,'extraction_tokens_out':0,
            'extraction_call_count':0,'retry_call_count':0,
            'extraction_elapsed_s':0.0,'field_revisions':[],'critical_fields_missing':[],
            'quality_flags':[f'VIRTUAL_COVER_INK_RATIO={lower_ink:.4f}'],'is_virtual_subdocument':True,
            'virtual_parent_doc_type':'TITRE_TRAVAIL','frontiere_planche':crops['frontiere'],
        })
        additions.append(v)
    return records+additions


def _render_for_strategy(record,pdf_path,strategy_name,crop=None,max_side=None):
    if strategy_name=='STANDARD':
        return record['image']
    return render_page_region(pdf_path,record['page_num']-1,zoom=PDF_ZOOM_HAUTE_DEF,
                              max_side=max_side or IMAGE_MAX_SIZE_HAUTE_DEF,crop=crop)


def _initial_job(record,pdf_path):
    dt=record['doc_type']; prompt=PROMPTS_EXTRACTION[dt]
    if dt=='TITRE_TRAVAIL':
        crops=crops_planche_permis(record['image']); record['frontiere_planche']=crops['frontiere']
        return {'record':record,'prompt':prompt,'image':_render_for_strategy(record,pdf_path,'HD_BLOC_TITRE',crops['titre']),
                'strategy':'HD_BLOC_TITRE','profile':'HD','allowed_fields':None,'overwrite':False,
                'max_new_tokens':MAX_NEW_TOKENS_EXTRACTION}
    if dt=='PERMIS_TRAVAIL_COUVERTURE':
        if record.get('is_virtual_subdocument'):
            crops=crops_planche_permis(record['image']); record['frontiere_planche']=crops['frontiere']
            crop=crops['couverture']; name='HD_BLOC_COUVERTURE'
        else:
            crop=(0,1,0,1); name='PAGE_ENTIERE_HD'
        return {'record':record,'prompt':prompt,'image':_render_for_strategy(record,pdf_path,name,crop),
                'strategy':name,'profile':'HD','allowed_fields':None,'overwrite':False,
                'max_new_tokens':MAX_NEW_TOKENS_EXTRACTION}
    return {'record':record,'prompt':prompt,'image':record['image'],'strategy':'STANDARD','profile':'STANDARD',
            'allowed_fields':None,'overwrite':False,'max_new_tokens':MAX_NEW_TOKENS_EXTRACTION}


def _merge_job_result(job,output):
    record=job['record']; dt=record['doc_type']
    parsed=parse_json_response(output.get('text',''))  # aucune normalisation / nettoyage de valeur
    expected=CHAMPS_ATTENDUS.get(dt) or []
    allowed=set(job['allowed_fields']) if job.get('allowed_fields') is not None else None
    added=corrected=0

    # Une entrée = un véritable appel Qwen d'extraction.
    # On évite de recopier parsed_data + raw_data + extraction_raw_text trois fois.
    attempt={
        'strategy':job['strategy'],
        'fields_requested':sorted(allowed) if allowed is not None else None,
        'fields_returned':sorted([k for k,v in parsed.items() if k in expected and not is_missing_raw(v)]),
        'tokens_in':int(output.get('tokens_in',0) or 0),
        'tokens_out':int(output.get('tokens_out',0) or 0),
        'elapsed_s':float(output.get('elapsed_s',0) or 0),
        'is_retry': bool(job.get('overwrite')),
    }
    if STORE_QWEN_RAW_TEXT_IN_ATTEMPTS:
        attempt['raw_text']=output.get('text')
    if STORE_PARSED_DATA_IN_ATTEMPTS:
        attempt['parsed_data']=parsed
    record['extraction_attempts'].append(attempt)
    record['extraction_call_count']=int(record.get('extraction_call_count',0) or 0)+1
    if job.get('overwrite'):
        record['retry_call_count']=int(record.get('retry_call_count',0) or 0)+1

    for field,value in parsed.items():
        if field not in expected or (allowed is not None and field not in allowed):
            continue
        if is_missing_raw(value):
            continue
        old=record['raw_data'].get(field)
        if is_missing_raw(old):
            record['raw_data'][field]=value; added+=1
        elif job.get('overwrite') and str(old)!=str(value):
            record['raw_data'][field]=value; corrected+=1
            record['field_revisions'].append({'field':field,'old':old,'new':value,'strategy':job['strategy']})

    attempt['fields_added']=added
    attempt['fields_corrected']=corrected

    record['extraction_tokens_in']+=int(output.get('tokens_in',0) or 0)
    record['extraction_tokens_out']+=int(output.get('tokens_out',0) or 0)
    record['extraction_elapsed_s']=round(record.get('extraction_elapsed_s',0)+float(output.get('elapsed_s',0) or 0),3)
    if STORE_AGGREGATED_EXTRACTION_RAW_TEXT:
        block=f"[{job['strategy']}] {output.get('text','')}"
        record['extraction_raw_text']=(record['extraction_raw_text']+'\n\n'+block) if record.get('extraction_raw_text') else block
    else:
        record['extraction_raw_text']=None

    taux=taux_remplissage(record['raw_data'],expected)
    record['extraction_taux_remplissage']=taux
    record['extraction_strategies'].append({
        'nom':job['strategy'],
        'fields_requested':sorted(allowed) if allowed else None,
        'champs_ajoutes':added,
        'champs_corriges':corrected,
        'taux_apres':taux,
        'is_retry':bool(job.get('overwrite')),
    })


def _run_job_chunk(chunk):
    if not chunk: return
    max_new=max(int(j.get('max_new_tokens',MAX_NEW_TOKENS_EXTRACTION)) for j in chunk)
    try:
        outs=ask_batch_mixed([j['prompt'] for j in chunk],[j['image'] for j in chunk],max_new)
        for j,o in zip(chunk,outs): _merge_job_result(j,o)
    except Exception as exc:
        if len(chunk)>1:
            if is_cuda_oom(exc):
                print(f'⚠️ OOM batch {len(chunk)} -> découpage'); gc.collect(); torch.cuda.empty_cache()
            else:
                print(f'⚠️ Batch {len(chunk)} refusé ({type(exc).__name__}) -> sous-batches')
            mid=len(chunk)//2; _run_job_chunk(chunk[:mid]); _run_job_chunk(chunk[mid:]); return
        raise


def run_extraction_jobs(jobs):
    std=[j for j in jobs if j.get('profile')=='STANDARD']
    hd=[j for j in jobs if j.get('profile')!='STANDARD']
    for s in range(0,len(std),GPU_BATCH_SIZE_EXTRACTION_STANDARD): _run_job_chunk(std[s:s+GPU_BATCH_SIZE_EXTRACTION_STANDARD])
    for s in range(0,len(hd),GPU_BATCH_SIZE_EXTRACTION_HD): _run_job_chunk(hd[s:s+GPU_BATCH_SIZE_EXTRACTION_HD])


def _fields_needing_retry(record):
    dt=record['doc_type']; expected=CHAMPS_ATTENDUS.get(dt) or []; data=record.get('raw_data') or {}
    critical=critical_problems(record)
    missing=[f for f in expected if is_missing_raw(data.get(f))]
    fields=list(critical)
    if taux_remplissage(data,expected)<SEUIL_REMPLISSAGE_OK:
        for f in missing:
            if f not in fields: fields.append(f)
            if len(fields)>=MAX_TARGETED_FIELDS: break
    return fields[:MAX_TARGETED_FIELDS]


def build_targeted_retry_jobs(records,pdf_path):
    if not ENABLE_TARGETED_RETRY: return []
    jobs=[]
    for r in records:
        dt=r.get('doc_type')
        if dt not in PROMPTS_EXTRACTION: continue
        fields=_fields_needing_retry(r)
        if not fields: continue
        if dt=='TITRE_TRAVAIL':
            crops=crops_planche_permis(r['image'])
            identity=[f for f in fields if f in TITLE_IDENTITY_FIELDS]
            post=[f for f in fields if f in TITLE_POST_FIELDS]
            if identity:
                jobs.append({'record':r,'prompt':build_targeted_prompt(dt,identity),
                             'image':_render_for_strategy(r,pdf_path,'HD_COLONNE_IDENTITE',crops['colonne_identite']),
                             'strategy':'HD_COLONNE_IDENTITE_TARGETED','profile':'HD','allowed_fields':identity,
                             'overwrite':True,'max_new_tokens':MAX_NEW_TOKENS_TARGETED})
            if post:
                jobs.append({'record':r,'prompt':build_targeted_prompt(dt,post),
                             'image':_render_for_strategy(r,pdf_path,'HD_COLONNE_POSTE',crops['colonne_poste']),
                             'strategy':'HD_COLONNE_POSTE_TARGETED','profile':'HD','allowed_fields':post,
                             'overwrite':True,'max_new_tokens':MAX_NEW_TOKENS_TARGETED})
            continue
        if dt=='PERMIS_TRAVAIL_COUVERTURE':
            crop=crops_planche_permis(r['image'])['couverture'] if r.get('is_virtual_subdocument') else (0,1,0,1)
            jobs.append({'record':r,'prompt':build_targeted_prompt(dt,fields),
                         'image':_render_for_strategy(r,pdf_path,'PERMIS_RETRY_HD',crop,IMAGE_MAX_SIZE_PERMIS_RETRY),
                         'strategy':'PERMIS_RETRY_HD_TARGETED','profile':'HD','allowed_fields':fields,
                         'overwrite':True,'max_new_tokens':MAX_NEW_TOKENS_TARGETED})
            continue
        crop=(0.03,0.80,0,1) if dt in {'CONTRAT_TRAVAIL','CONTRAT_SPECIFIQUE'} else (0,1,0,1)
        jobs.append({'record':r,'prompt':build_targeted_prompt(dt,fields),
                     'image':_render_for_strategy(r,pdf_path,'TARGETED_HD',crop),
                     'strategy':'TARGETED_HD','profile':'HD','allowed_fields':fields,
                     'overwrite':True,'max_new_tokens':MAX_NEW_TOKENS_TARGETED})
    return jobs


def build_final_safety_jobs(records,pdf_path):
    if not ENABLE_FINAL_HD_SAFETY: return []
    jobs=[]
    for r in records:
        dt=r.get('doc_type')
        if dt not in PROMPTS_EXTRACTION: continue
        problems=critical_problems(r)
        if not problems: continue
        if dt=='TITRE_TRAVAIL':
            crop=crops_planche_permis(r['image'])['titre']; max_side=IMAGE_MAX_SIZE_PERMIS_RETRY; strategy='HD_SAFETY_TITRE_2000'
        elif dt=='PERMIS_TRAVAIL_COUVERTURE':
            crop=crops_planche_permis(r['image'])['couverture'] if r.get('is_virtual_subdocument') else (0,1,0,1)
            max_side=IMAGE_MAX_SIZE_PERMIS_RETRY; strategy='HD_SAFETY_PERMIS_2000'
        elif dt in {'CONTRAT_TRAVAIL','CONTRAT_SPECIFIQUE'}:
            crop=(0.03,0.82,0,1); max_side=IMAGE_MAX_SIZE_HAUTE_DEF; strategy='HD_SAFETY_CENTRAL'
        else:
            crop=(0,1,0,1); max_side=IMAGE_MAX_SIZE_HAUTE_DEF; strategy='HD_SAFETY_FULL'
        jobs.append({'record':r,'prompt':PROMPTS_EXTRACTION[dt],
                     'image':_render_for_strategy(r,pdf_path,strategy,crop,max_side),
                     'strategy':strategy,'profile':'HD','allowed_fields':problems,
                     'overwrite':True,'max_new_tokens':MAX_NEW_TOKENS_EXTRACTION})
    return jobs


def finalize_extraction_record(record):
    dt=record.get('doc_type')
    if dt not in PROMPTS_EXTRACTION:
        record['extraction_status']='NON_APPLICABLE'; return record
    expected=CHAMPS_ATTENDUS.get(dt) or []
    for f in expected:
        record['raw_data'].setdefault(f,None)
    record['extraction_taux_remplissage']=taux_remplissage(record['raw_data'],expected)
    problems=critical_problems(record); record['critical_fields_missing']=problems
    flags=list(record.get('quality_flags') or [])
    if problems: flags.append('CRITICAL_MISSING_OR_STRUCTURALLY_SUSPECT')
    if record['extraction_taux_remplissage']<SEUIL_REMPLISSAGE_MIN: flags.append('LOW_FILL_RATE')
    if record.get('field_revisions'): flags.append('FIELD_REREAD_BY_HD_RETRY')
    record['quality_flags']=list(dict.fromkeys(flags))
    if not any(not is_missing_raw(v) for v in record['raw_data'].values()): record['extraction_status']='JSON_VIDE'
    elif problems or record['extraction_taux_remplissage']<SEUIL_REMPLISSAGE_MIN: record['extraction_status']='PARTIELLE'
    else: record['extraction_status']='OK'
    return record


def extract_classified_pages(records,pdf_path):
    applicable=[r for r in records if r.get('doc_type') in PROMPTS_EXTRACTION]

    # 1) Un seul appel initial par document détecté.
    # Pour TITRE_TRAVAIL, cet appel initial reste volontairement HD : c'est le niveau
    # de qualité validé depuis la V7.2 pour lire correctement le permis/titre.
    run_extraction_jobs([_initial_job(r,pdf_path) for r in applicable])

    # 2) Retry uniquement si des champs critiques sont absents/suspects, ou si le
    # taux de remplissage est réellement insuffisant. Un document complet ne repasse pas.
    targeted=build_targeted_retry_jobs(applicable,pdf_path)
    run_extraction_jobs(targeted)

    # 3) Dernière sécurité uniquement si un problème critique subsiste après le retry ciblé.
    safety=build_final_safety_jobs(applicable,pdf_path)
    run_extraction_jobs(safety)

    for r in records:
        finalize_extraction_record(r)
    return records


def print_call_diagnostics(records):
    if not PRINT_CALL_DIAGNOSTICS:
        return
    print('\n--- Diagnostic appels Qwen par page ---')
    for r in records:
        if r.get('doc_type') not in PROMPTS_EXTRACTION:
            continue
        strategies=[a.get('strategy') for a in (r.get('extraction_attempts') or [])]
        print(
            f"page={r.get('page_num')} type={r.get('doc_type')} "
            f"classification_calls={len(r.get('classification_attempts') or [])} "
            f"extraction_calls={r.get('extraction_call_count',0)} "
            f"retries={r.get('retry_call_count',0)} "
            f"fill={r.get('extraction_taux_remplissage')} "
            f"critical={r.get('critical_fields_missing')} "
            f"strategies={strategies}"
        )


def _json_safe_record(record):
    return {k:v for k,v in record.items() if k!='image'}


def process_pdf(pdf_path):
    t0=time.time(); log(f'📁 {pdf_path.name}')
    pages=pdf_to_pages(pdf_path)
    records=add_virtual_permit_subdocuments(classify_pages(pages))
    records=extract_classified_pages(records,pdf_path)
    print_call_diagnostics(records)
    tokens_in=sum(r.get('classification_tokens_in',0)+r.get('extraction_tokens_in',0) for r in records)
    tokens_out=sum(r.get('classification_tokens_out',0)+r.get('extraction_tokens_out',0) for r in records)
    elapsed=round(time.time()-t0,3)
    dossier={
        'schema_version':SCHEMA_VERSION,
        'field_schema_hash':FIELD_SCHEMA_HASH,
        'field_schema':FIELD_SCHEMA,
        'source_file':pdf_path.name,
        'source_sha256':sha256_file(pdf_path),
        'pipeline_version':PIPELINE_VERSION,
        'extraction_engine':{
            'model':'Qwen3.6-27B-FP8','model_path':MODEL_PATH,'flash_attn':False,
            'standard_max_side':IMAGE_MAX_SIZE,'classification_max_side':IMAGE_MAX_SIZE_CLASSIFICATION,
            'hd_max_side':IMAGE_MAX_SIZE_HAUTE_DEF,'permit_retry_max_side':IMAGE_MAX_SIZE_PERMIS_RETRY,
            'created_at':datetime.now().isoformat(timespec='seconds'),
        },
        'stats':{
            'pages':len(pages),'page_records':len(records),
            'virtual_subdocuments':sum(bool(r.get('is_virtual_subdocument')) for r in records),
            'classification_calls':sum(len(r.get('classification_attempts') or []) for r in records),
            'extraction_calls':sum(int(r.get('extraction_call_count',0) or 0) for r in records),
            'retry_calls':sum(int(r.get('retry_call_count',0) or 0) for r in records),
            'qwen_calls_total':sum(len(r.get('classification_attempts') or []) + int(r.get('extraction_call_count',0) or 0) for r in records),
            'extraction_passes':sum(len(r.get('extraction_strategies') or []) for r in records),
            'field_revisions':sum(len(r.get('field_revisions') or []) for r in records),
            'tokens_in':int(tokens_in),'tokens_out':int(tokens_out),'tokens_total':int(tokens_in+tokens_out),
            'elapsed_s':elapsed,
        },
        'page_records':[_json_safe_record(r) for r in records],
    }
    canonical_checkpoint_path(pdf_path).write_text(json.dumps(dossier,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    return dossier

print('✅ Moteur Extraction RAW V9 prêt')


## 11. Tests techniques sans inférence


In [ ]:
assert sum(len(v) for v in CHAMPS_ATTENDUS.values()) == 99
assert 'TTR_NUMERO_PERMIS' in CHAMPS_ATTENDUS['TITRE_TRAVAIL']
assert IMAGE_MAX_SIZE_HAUTE_DEF >= 1800
assert IMAGE_MAX_SIZE_PERMIS_RETRY >= 2000
assert value_is_suspect('DOM_SALAIRE_NET_MENSUEL','23.340.43') is False  # ne pas interpréter en Partie 1
# Un TITRE_TRAVAIL complet et structurellement valide ne doit pas être relu.
_test_title={'doc_type':'TITRE_TRAVAIL','raw_data':{k:'X' for k in CHAMPS_ATTENDUS['TITRE_TRAVAIL']}}
# rendre les champs structurés plausibles
_test_title['raw_data']['TTR_NUMERO_PERMIS']='20-00028138 / 31-26-000395'
assert _fields_needing_retry(_test_title) == []
print('✅ Tests structurels V9.1 Partie 1 OK')


## 12. Exécution — test sur 10 dossiers


In [ ]:
if not pdfs:
    print('⚠️ Aucun PDF dans', INPUT_DIR)
else:
    results=[]; errors=[]
    for i,pdf in enumerate(pdfs,1):
        print(f'\n[{i}/{len(pdfs)}] {pdf.name}')
        existing=load_existing_checkpoint(pdf)
        if existing is not None:
            print('↪ checkpoint V9.1 RAW valide réutilisé')
            d=existing; statut='REPRIS'
        else:
            try:
                d=process_pdf(pdf); statut='TRAITE'
            except Exception as exc:
                log(f'❌ {pdf.name} : {repr(exc)}')
                errors.append({'source_file':pdf.name,'error':repr(exc),'date':datetime.now().isoformat(timespec='seconds')})
                continue
        s=d.get('stats') or {}
        results.append({'source_file':d.get('source_file'),'status':statut,'sha256':d.get('source_sha256'),
                        'pages':s.get('pages'),'page_records':s.get('page_records'),
                        'tokens_total':s.get('tokens_total'),'elapsed_s':s.get('elapsed_s'),
                        'qwen_calls_total':s.get('qwen_calls_total'),'extraction_calls':s.get('extraction_calls'),'retry_calls':s.get('retry_calls'),
                        'json_path':str(canonical_checkpoint_path(pdf))})
        print(f"✅ {statut} | pages={s.get('pages')} | Qwen calls={s.get('qwen_calls_total')} | extraction={s.get('extraction_calls')} | retries={s.get('retry_calls')} | tokens={s.get('tokens_total')} | temps={s.get('elapsed_s')}s")

    manifest={'schema_version':SCHEMA_VERSION,'pipeline_version':PIPELINE_VERSION,
              'field_schema_hash':FIELD_SCHEMA_HASH,'generated_at':datetime.now().isoformat(timespec='seconds'),
              'results':results,'errors':errors}
    MANIFEST_PATH.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    pd.DataFrame(results).to_csv(INDEX_CSV_PATH,index=False,encoding='utf-8-sig')
    print('\n✅ Manifest :',MANIFEST_PATH)
    print('✅ Index    :',INDEX_CSV_PATH)
    print('✅ JSON RAW :',JSON_DIR)


## Contrat de sortie de la Partie 1

Le JSON RAW contient `raw_data` et les réponses Qwen (`extraction_raw_text`, `extraction_attempts`). Il ne contient volontairement **ni `normalized_data`, ni `dossier_row`, ni `planning_tl`**.

Une valeur telle que `23.340.43` reste `23.340.43` dans le RAW. La Partie 2 est seule autorisée à la normaliser.


### V9.1 — lecture des compteurs

- `classification_calls` : appels Qwen consacrés à la classification.
- `extraction_calls` : appels Qwen qui lisent effectivement les champs.
- `retry_calls` : sous-ensemble des appels d'extraction déclenchés après la première lecture.
- `qwen_calls_total` : classification + extraction.

`raw_data`, `extraction_attempts` et `extraction_strategies` ne signifient pas trois lectures : ils décrivent le résultat final et l'audit des mêmes appels.
